# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Daniela Renta**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [7]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [8]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)


    ##### EDITAR ESTA ZONA - USO HEURISTICA GREEDY CON H(n) 
    # de la clase que onifica con -1 por cada disco ubicado correctamente en la torre objetivo. #####

    import heapq
    import itertools

    def get_pegs(state):
        """
        Obtengo las tres torres del estado.
        Lo dejo así porque necesito mirar la torre destino
        para calcular la heurística.
        """
        if hasattr(state, "pegs"):
            return state.pegs
        elif hasattr(state, "rods"):
            return state.rods
        elif hasattr(state, "towers"):
            return state.towers
        else:
            return [state.tower_1, state.tower_2, state.tower_3]

    def heuristic(state):
        """
        Heurística del campus:
        h(n) bonifica con -1 por cada disco ubicado correctamente
        en la torre objetivo.
        Si con 5 discos tengo 3 bien ubicados:
        h(n) = -3
        """
        pegs = get_pegs(state)

        target_peg = tuple(pegs[2])
        goal_peg = tuple(range(number_disks, 0, -1))

        correct_disks = 0

        for current_disk, goal_disk in zip(target_peg, goal_peg):
            if current_disk == goal_disk:
                correct_disks += 1
            else:
                break

        return -correct_disks

    frontier = []
    explored = set()

    counter = itertools.count()
    initial_node = NodeHanoi(problem.initial)

    heapq.heappush(
        frontier,
        (heuristic(initial_node.state), next(counter), initial_node)
    )

    node_explored = 0
    max_depth = 0

    while len(frontier) != 0:
        _, _, node = heapq.heappop(frontier)

        if node.state in explored:
            continue

        node_explored += 1
        explored.add(node.state)

        max_depth = max(max_depth, node.depth)

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": float(node.state.accumulated_cost),
            }
            return node, metrics

        for next_node in node.expand(problem):
            if next_node.state not in explored:
                heapq.heappush(
                    frontier,
                    (heuristic(next_node.state), next(counter), next_node)
                )

    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": None,
    }

    #####

Se prueba la función:

In [9]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [10]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 116
states_visited: 116
nodes_in_frontier: 32
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [11]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [12]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
